# **Setup**

In [1]:
import os, sys

# Repository information
REPO_NAME = "RecSys-Challenge-2025"
REPO_URL  = f"github.com/Lv1g1/{REPO_NAME}.git"

# Detect environment
IS_COLAB = 'content' in os.getcwd()
IS_KAGGLE = 'kaggle' in os.getcwd()
IS_LOCAL = not (IS_COLAB or IS_KAGGLE)

WORKING_DIR = os.getcwd()

if IS_COLAB:
    WORKING_DIR = "/content"

    # Mount Google Drive
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

    # Get GitHub token via input
    def get_token():
        from getpass import getpass
        return getpass("GitHub Token: ")

elif IS_KAGGLE:
    WORKING_DIR = "/kaggle/working"

    # Get GitHub token from Kaggle secrets
    def get_token():
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("Token")

# If local environment assume inside the repo
LOCAL_REPO_PATH = "/home/luigi/RecSys" if IS_LOCAL else os.path.join(WORKING_DIR, REPO_NAME)

# Clone the repository if it doesn't exist
if not os.path.exists(LOCAL_REPO_PATH):
    os.chdir(WORKING_DIR)
    token = get_token()

    !git clone https://{token}@{REPO_URL}
else:
    print("Repo already exists — pulling latest changes")
    os.chdir(LOCAL_REPO_PATH)
    !git pull
    os.chdir(WORKING_DIR)

# Add to Python PATH
if LOCAL_REPO_PATH not in sys.path:
    sys.path.append(LOCAL_REPO_PATH)

Repo already exists — pulling latest changes
Already up to date.


In [2]:
if IS_COLAB or False:  # Set to True if you want to recompile Cython files
    os.chdir(LOCAL_REPO_PATH)
    !python run_compile_all_cython.py
    os.chdir(WORKING_DIR)

In [3]:
if IS_COLAB:
    !pip install optuna

import optuna

In [4]:
import importlib
import numpy as np

from Challenge import paths
importlib.reload(paths)

from Challenge.hyper_tuning import ModelOptimizer

Running on kaggle — storage at: /kaggle/working
Running on kaggle — storage at: /kaggle/working


/kaggle/working/RecSys-Challenge-2025/Challenge/hyper_tuning.py:75: ExperimentalWarning: WilcoxonPruner is experimental (supported from v3.6.0). The interface can change in the future.
  def create_study(self, study_name, direction="maximize", load_if_exists=True, pruner=optuna.pruners.WilcoxonPruner()):
/kaggle/working/RecSys-Challenge-2025/Challenge/hyper_tuning.py:107: ExperimentalWarning: WilcoxonPruner is experimental (supported from v3.6.0). The interface can change in the future.
  def create_and_optimize_study(self, study_name, objective_function, n_trials=50, direction="maximize", load_if_exists=True, pruner=optuna.pruners.WilcoxonPruner()):


# **Load Data**

In [5]:
# Load datasets
folds = paths.load_cv_folds(k=5)

In [6]:
def evaluate_recommender(recommender, at, URM_validation):
    cumulative_recall = 0.0
    num_eval = 0
    
    for user_id in range(URM_validation.shape[0]):
        relevant_items = URM_validation.indices[URM_validation.indptr[user_id]:URM_validation.indptr[user_id+1]]
        
        if len(relevant_items)>0:
            num_eval+=1
            
            recommended_items = recommender.recommend(user_id, cutoff=at)
            
            is_relevant = np.isin(recommended_items, relevant_items, assume_unique=True)
            recall_score = np.sum(is_relevant, dtype=np.float32) / relevant_items.shape[0]

            cumulative_recall += recall_score

    return cumulative_recall / num_eval

# **Hyperparameter search**

In [7]:
from Recommenders.KNN.UserKNNCFRecommender import UserKNNCFRecommender

optimizer = ModelOptimizer("UserKNN_asymmetric")

STUDY_NAME = UserKNNCFRecommender.RECOMMENDER_NAME + '_asymmetric'

In [8]:
def objective_function(optuna_trial: optuna.trial.Trial) -> float:
    params = {
        "similarity": "asymmetric",
        "topK": optuna_trial.suggest_int("topK", 5, 500),
        "shrink": optuna_trial.suggest_int("shrink", 0, 1000),
        "normalize": optuna_trial.suggest_categorical("normalize", [True, False]),
        "asymmetric_alpha": optuna_trial.suggest_float("asymmetric_alpha", 0.0, 1.0),
        "feature_weighting": optuna_trial.suggest_categorical("feature_weighting", ["none", "TF-IDF", "BM25"]),
    }

    if params["feature_weighting"] == "BM25":
        params["BM25_k1"] = optuna_trial.suggest_float("BM25_k1", 0.5, 2.0)
        params["BM25_b"] = optuna_trial.suggest_float("BM25_b", 0.0, 1.0)
    
    validation_scores = []
    for fold_idx, (URM_train, URM_validation) in enumerate(folds):
        # Train the recommender
        recommender_instance = UserKNNCFRecommender(URM_train)
        recommender_instance.fit(**params)
        
        # Evaluate
        score = evaluate_recommender(recommender_instance, at=20, URM_validation=URM_validation)
        validation_scores.append(score)
        
        # Show fold result
        print(f"  Fold {fold_idx+1}/{len(folds)} - Score: {score}")

        # Report intermediate result to Optuna
        optuna_trial.report(score, fold_idx)

        # Ask Optuna to prune if performance is poor
        if optuna_trial.should_prune():
            # Return the average score so far instead of raising TrialPruned,
            # which is a common workaround for WilcoxonPruner.
            return np.mean(validation_scores)
        
    # Log folds performance
    optimizer.log_folds(validation_scores, params)

    # Return the mean CV score for the fully completed trial
    return np.mean(validation_scores)

In [10]:
optuna_study = optimizer.create_and_optimize_study(
    study_name=STUDY_NAME,
    objective_function=objective_function,
    n_trials=36
)

[I 2025-11-20 16:56:49,557] Using an existing study with name 'UserKNNCFRecommender_asymmetric' instead of creating a new one.


  0%|          | 0/36 [00:00<?, ?it/s]

Similarity column 27095 (100.0%), 600.89 column/sec. Elapsed time 45.09 sec
  Fold 1/5 - Score: 0.1737661362822069
Similarity column 27095 (100.0%), 604.07 column/sec. Elapsed time 44.85 sec
  Fold 2/5 - Score: 0.1730484375645972
Similarity column 27095 (100.0%), 603.57 column/sec. Elapsed time 44.89 sec
  Fold 3/5 - Score: 0.17388059332942457
Similarity column 27095 (100.0%), 604.15 column/sec. Elapsed time 44.85 sec
  Fold 4/5 - Score: 0.17542707953325776
[I 2025-11-20 17:01:20,355] Trial 69 finished with value: 0.17403056167737163 and parameters: {'topK': 413, 'shrink': 211, 'normalize': True, 'asymmetric_alpha': 0.15440878876656977, 'feature_weighting': 'none'}. Best is trial 67 with value: 0.2351886911907281.
Similarity column 27095 (100.0%), 606.66 column/sec. Elapsed time 44.66 sec
  Fold 1/5 - Score: 0.2301725674179274
Similarity column 27095 (100.0%), 605.99 column/sec. Elapsed time 44.71 sec
  Fold 2/5 - Score: 0.23057074066635036
Similarity column 27095 (100.0%), 610.76 colu

In [9]:
optuna_study = optimizer.create_study(STUDY_NAME)

[I 2025-11-21 11:09:04,478] Using an existing study with name 'UserKNNCFRecommender_asymmetric' instead of creating a new one.


In [10]:
optuna.visualization.plot_optimization_history(optuna_study)

In [11]:
optuna.visualization.plot_param_importances(optuna_study)

In [12]:
optuna.visualization.plot_parallel_coordinate(optuna_study)

## **Best Model**
Best Value: 0.2358744211659045

Best Params: {'topK': 418, 'shrink': 137, 'normalize': True, 'asymmetric_alpha': 0.2887952454899258, 'feature_weighting': 'BM25', 'BM25_k1': 1.1959159850418524, 'BM25_b': 0.0998562786186147}